# Data Analysis Notes

*Riddhi Patel* <br>
*March 2026*

Welcome to Riddhi's notes on imaging data analysis! I am simply radiating with joy to be on this journey with you...I wonder if JWST can detect it :,)



---
## Contents
* [Setup](#setup)
    * [Setting up Virtual Environment](#venv)
    * [Install Python Packages](#python)
* [Image Combination](#image_combo)
    * [Import and Display Data](#hdr)
    * [Display Image](#img)
    * [Pixel Values](#pix)
    * [Combining Images](#combo)
* [Data Management](#data_management)
* [Overscan Subtract and Trim](#ot)
* [Bias Correction](#otz)
* [Darks Analysis](#darks)
* [DS9 Notes](#ds9)


## Setup <a class="anchor" id="setup"></a>

### Setting up Virtual Environment <a class="anchor" id="venv"></a>

I know how this looks...I know you're saying, "Riddhi, everybody can make a virtual environment. Surely this doesn't need to be here." Well, yes and no. Personally, I am horrible at recall and need to write down the steps so I can do it myself when Suyash isn't sitting 10 ft away from me. <br>

    $ python -m venv [virtual environment name].venv
    $ source [virtual environment name].venv/bin/activate


### Install Python Packages <a class="anchor" id="venv"></a>

There specific imaging packages in python that are very helpful to pip install to your virtual environment that I will forget if I have to do it myself so I am putting it all here!

    $ pip install numpy astropy ccdproc photutils ipywidgets matplotlib 



In [2]:
# IMPORT BLOCK

import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt
from matplotlib import rc
%matplotlib inline
from astropy.visualization import hist, ZScaleInterval
from ccdproc import ImageFileCollection
import ccdproc as ccdp
from astropy.modeling import fitting
from astropy.modeling.models import Polynomial1D,Chebyshev1D,Legendre1D,Hermite1D
from astropy.nddata import CCDData
from astropy import units as u
from astropy.stats import sigma_clip, mad_std
import glob
import os

## Image Combination <a class="anchor" id="image_combo"></a>

### Import and Display Data <a class="anchor" id="hdr"></a>

The code below will help you open, display, and use header and data information from a fits file. From the header, you can check the size of the image, bits per pixel, airmass, exposure time, etc. 

In [3]:
hdu = fits.open('[enter file path]') #imports fits file

hdr = hdu[0].header #lists important header information 

img = hdu[0].data #lists data as numpy array 
img = fits.getdata('[enter file path]') #lists data as numpy array
data, header = fits.getdata('[enter file path]', header=True) #lists data as numpy array and header info (SUPER USEFUL!)

img.shape #gives us dimensions of numpy array image (good for sanity-checking after trimming)
np.max(img) #gives us max value in the numpy array image data (max val typically white)

FileNotFoundError: [Errno 2] No such file or directory: '[enter file path]'

### Display Image <a class="anchor" id="img"></a>

The code below will help you display the pretty picture stored in a fits file! 

In [ ]:
img = fits.getdata('[fits file path]')

interval = ZScaleInterval()
vmin, vmax = interval.get_limits(img)

plt.imshow(img, vmin=vmin, vmax=vmax) #vmin and vmax tells us what value to assign white/black to! White will be assigned to vmax.
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title('[name of object]')
plt.show()

### Pixel Values <a class="anchor" id="pix"></a>

The code below will help you analyze the pixels and their values (from columns/rows of the image as well)! 

In [ ]:
## CREATES A SCATTER PLOT OF IMAGE VALUES BY PIXEL NUMBER
hdu = fits.open('[fits file path]')
hdr = hdu[0].header
img = hdu[0].data

img_1_row = img[1, :]
img_1_col = img[:, 1]

selected = img_1_col
pixel_num = np.arange(0, len(selected), 1)

plt.scatter(pixel_num, selected, s=1)
plt.xlabel('Pixel Number')
plt.ylabel('Pixel Value')
plt.show()

## CREATES A HISTOGRAM OF PIXEL VALUES IN A FITS FILE
hdu = fits.open('[fits file path]')
hdr = hdu[0].header
bias = hdu[0].data

low_limit = 0
hi_limit = 800

bins = np.histogram_bin_edges(bias, bins=100, range=[low_limit,hi_limit])

plt.figure(figsize=(6,6))
hist(bias.flatten(), bins=bins, alpha=0.5, label='bias',rwidth=1)
plt.legend()
plt.xlim([low_limit,hi_limit])
plt.xlabel('Counts')
plt.ylabel('Number of pixels')
plt.show()

### Combining Images <a class="anchor" id="combo"></a>

There are many ways to combine images (using mean, median, etc). Below, there is code to do mean and median image combination using ccdproc!

There's also code included to do this for a small number of images below that in case it's useful later, but ccdproc does this in much less steps.

In [ ]:
## CREATES MASTER BIAS FILE AND SAVES TO OUTPUT DIRECTORY

bias_ot_files = sorted(
    glob.glob('enter file path') 
)
print('The number of biases we have is:', len(bias_ot_files))

master_bias = ccdp.combine(bias_ot_files,
                           unit='adu',
                           method='average',
                           sigma_clip=False)
master_bias_array = np.array(master_bias)
output_path = os.path.join(output_dir, 'master_bias.fits')
fits.writeto(output_path, master_bias_array, overwrite=True)

In [ ]:
hdu1 = fits.open('[fits file path 1]')
hdu2 = fits.open('[fits file path 2]')
hdu3 = fits.open('[fits file path 3]')

img1 = hdu1[0].data
img2 = hdu2[0].data
img3 = hdu3[0].data

mean_image = np.mean([img1, img2, img3], axis=0)
median_image = np.median([img1, img2, img3], axis=0)

interval = ZScaleInterval()
mean_min, mean_max = interval.get_limits(mean_image)
median_max, median_max = interval.get_limits(median_image)

plt.imshow(img, vmin=mean_min, vmax=mean_max, cmap="gray") #vmin and vmax tells us what value to assign white/black to! White will be assigned to vmax.
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title('M101')
plt.show()

## Data Management <a class="anchor" id="data_management"></a>

This section is about how to manage lots of data files at once. Personally, I really like glob so I'll show examples from HW4 on how to use it properly to grab the files you need faster than just copying and pasting a bunch of times! It's also helpful to convert this list of files into numpy arrays for later pixel analysis, so I've included that section as well!

In [ ]:
## USES GLOB TO PULL BIASES, FLATS, DARKS, AND PG1633/NGC6823 IMAGES

data_dir = '/Users/pater32/Documents/GitHub/astr_8060_s26/work/ripatel/image_combination/Imaging/'

bias_files = sorted(
    glob.glob(data_dir+'a09[3-9].fits') + #93-99
    glob.glob(data_dir+'a10[0-9].fits') + #100-109
    glob.glob(data_dir+'a11[0-1].fits')   #110-111
)
print('The number of biases we have is:', len(bias_files))


flat_files = sorted(
    glob.glob(data_dir+'a00[5-9].fits') + #5-9
    glob.glob(data_dir+'a01[0-9].fits') + #10-19
    glob.glob(data_dir+'a02[0-9].fits') + #20-29
    glob.glob(data_dir+'a03[0-9].fits') + #30-39
    glob.glob(data_dir+'a04[0-9].fits') + #40-49
    glob.glob(data_dir+'a05[0-9].fits') + #50-59
    glob.glob(data_dir+'a06[0-2].fits')   #60-62
)
print('The number of flats we have is:', len(flat_files))


dark_files = sorted(
    glob.glob(data_dir+'d00[1-9].fits') + #1-9
    glob.glob(data_dir+'d01[0-5].fits')  #10-15
)
print('The number of darks we have is:', len(dark_files))


pg_files = sorted(
    glob.glob(data_dir+'a06[4-9].fits') + #64-69
    glob.glob(data_dir+'a07[0-9].fits') + #70-79
    glob.glob(data_dir+'a08[0-9].fits') + #80-89
    glob.glob(data_dir+'a09[0-2].fits') + #90-92
    glob.glob(data_dir+'a13[0-9].fits') + #130-139
    glob.glob(data_dir+'a20[2-9].fits') + #202-209
    glob.glob(data_dir+'a21[0-1].fits') + #210-211
    glob.glob(data_dir+'a22[1-9].fits') + #221-229 (We don't have any of these)
    glob.glob(data_dir+'a23[0-9].fits') + #230-239 (We only have #237-239)
    glob.glob(data_dir+'a24[0-6].fits')   #240-246
)
print('The number of PG1633 images we have is:', len(pg_files))


ngc_files = sorted(
    glob.glob(data_dir+'a15[3-9].fits') + #153-159
    glob.glob(data_dir+'a16[0-3].fits')  #160-163
)
print('The number of NGC6823 images we have is:', len(ngc_files))



In [ ]:
## MAKES ALL OF OUR FILE LISTS INTO NUMPY ARRAY CUBES

bias_array = np.array([fits.getdata(bias) for bias in bias_files])
bias_biassec = np.array([np.concatenate((bias[:, 0:53], bias[:, 2101:]), axis=1) for bias in bias_array])
bias_trimsec = np.array([bias[:, 53:2101] for bias in bias_array])
print('Bias array shape:', bias_array.shape,
      'Bias biassec array shape:', bias_biassec.shape,
      'Bias trimsec array shape:', bias_trimsec.shape)

flat_array = np.array([fits.getdata(flat) for flat in flat_files])
flat_biassec = np.array([np.concatenate((flat[:, 0:53], flat[:, 2101:]), axis=1) for flat in flat_array])
flat_trimsec = np.array([flat[:, 53:2101] for flat in flat_array])
print('Flat array shape:', flat_array.shape,
      'Flat biassec array shape:', flat_biassec.shape,
      'Flat trimsec array shape:', flat_trimsec.shape)

dark_array = np.array([fits.getdata(dark) for dark in dark_files])
dark_biassec = np.array([np.concatenate((dark[:, 0:53], dark[:, 2101:]), axis=1) for dark in dark_array])
dark_trimsec = np.array([dark[:, 53:2101] for dark in dark_array])
print('Dark array shape:', dark_array.shape,
      'Dark biassec array shape:', dark_biassec.shape,
      'Dark trimsec array shape:', dark_trimsec.shape)

pg_array = np.array([fits.getdata(img) for img in pg_files])
pg_biassec = np.array([np.concatenate((img[:, 0:53], img[:, 2101:]), axis=1) for img in pg_array])
pg_trimsec = np.array([img[:, 53:2101] for img in pg_array])
print('PG array shape:', pg_array.shape,
      'PG biassec array shape:', pg_biassec.shape,
      'PG trimsec array shape:', pg_trimsec.shape)

ngc_array = np.array([fits.getdata(img) for img in ngc_files])
ngc_biassec = np.array([np.concatenate((img[:, 0:53], img[:, 2101:]), axis=1) for img in ngc_array])
ngc_trimsec = np.array([img[:, 53:2101] for img in ngc_array])
print('NGC array shape:', ngc_array.shape,
      'NGC biassec array shape:', ngc_biassec.shape,
      'NGC trimsec array shape:', ngc_trimsec.shape)



## Overscan Subtract and Trim <a class="anchor" id="ot"></a>

In [ ]:
## DECIDE WHICH DEGREE POLYNOMIAL TO FIT TO THE OVERSCAN REGION

hdu = fits.open(data_dir+'a130.fits')
img = hdu[0].data
img_overscan = np.concatenate((img[:, 0:53], img[:, 2101:2200]), axis=1)

row_range = np.arange(0, len(img_overscan), 1)
means = []
for row in row_range:
    mean = np.mean(img_overscan[row, :])
    means.append(mean)

degree = 3
poly_model_coeff = np.polyfit(row_range, means, degree)
poly_model = np.poly1d(poly_model_coeff)

chebyshev_model = np.polynomial.Chebyshev.fit(row_range, means, deg=degree)
legendre_model = np.polynomial.Legendre.fit(row_range, means, deg=degree)
hermite_model = np.polynomial.Hermite.fit(row_range, means, deg=degree)

plt.scatter(row_range, means, s=2, label='Mean Values')
plt.plot(row_range, poly_model(row_range), label='Polynomial Fit', color='black')
plt.plot(row_range, chebyshev_model(row_range), label='Chebyshev Fit', color='orange')
plt.plot(row_range, legendre_model(row_range), label='Legendre Fit', color='red')
plt.plot(row_range, hermite_model(row_range), label='Hermite Fit', color='green')
plt.plot()
plt.xlabel('Row Number')
plt.ylabel('Mean Pixel Value')
plt.title('Mean Pixel Value by Row')
plt.legend()
plt.show()

In [ ]:
## OVERSCAN SUBTRACT AND TRIM EVERY FILE

##This section overscan subtracts and trims one bias file to make sure the for loops below are correct!
# bias_img = fits.getdata(data_dir+'a130.fits')
# ccd = CCDData(bias_img, unit=u.adu)
# overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
# overscan_ccd = CCDData(overscan, unit=ccd.unit)

# bias_img_o = ccdp.subtract_overscan(ccd, 
#                             overscan = overscan_ccd, 
#                             overscan_axis = 1,
#                             model = Polynomial1D(degree=3)
#                             ).data
# bias_img_ot = bias_img_o[:, 53:2101]
# bias_img_t = bias_img[:, 53:2101]

# print(np.mean(bias_img), np.mean(bias_img_o), np.mean(overscan))

# interval = ZScaleInterval()
# vmin, vmax = interval.get_limits(bias_img)
# plt.imshow(bias_img, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title('Original Image')
# plt.show()

# vmin, vmax = interval.get_limits(bias_img_o)
# plt.imshow(bias_img_o, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title('Overscan Subtracted')
# plt.show()

# vmin, vmax = interval.get_limits(bias_img_t)
# plt.imshow(bias_img_t, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title('Trimmed')
# plt.show()

# vmin, vmax = interval.get_limits(bias_img_ot)
# plt.imshow(bias_img_ot, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title('OS Sub + Trim')
# plt.show()

#BIASES     
bias_o_array = []
bias_ot_array = []

for bias in bias_files:
    bias_img, header = fits.getdata(bias, header=True)
    ccd = CCDData(bias_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    bias_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    bias_img_ot = bias_img_o[:, 53:2101]

    bias_o_array.append(bias_img_o)
    bias_ot_array.append(bias_img_ot)

    original_name = os.path.basename(bias)
    new_name = os.path.splitext(original_name)[0]+'_ot.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, bias_img_ot, header, overwrite=True)




#FLATS     
flat_o_array = []
flat_ot_array = []

for flat in flat_files:
    flat_img, header = fits.getdata(flat, header=True)
    ccd = CCDData(flat_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    flat_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    flat_img_ot = flat_img_o[:, 53:2101]

    flat_o_array.append(flat_img_o)
    flat_ot_array.append(flat_img_ot)

    original_name = os.path.basename(flat)
    new_name = os.path.splitext(original_name)[0]+'_ot.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, flat_img_ot, header, overwrite=True)


#DARKS     
dark_o_array = []
dark_ot_array = []

for dark in dark_files:
    dark_img, header = fits.getdata(dark, header=True)
    ccd = CCDData(dark_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    dark_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    dark_img_ot = dark_img_o[:, 53:2101]

    dark_o_array.append(dark_img_o)
    dark_ot_array.append(dark_img_ot)

    original_name = os.path.basename(dark)
    new_name = os.path.splitext(original_name)[0]+'_ot.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, dark_img_ot, header, overwrite=True)


#PG1633    
pg_o_array = []
pg_ot_array = []

for img in pg_files:
    pg_img, header = fits.getdata(img, header=True)
    ccd = CCDData(pg_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    pg_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    pg_img_ot = pg_img_o[:, 53:2101]

    pg_o_array.append(pg_img_o)
    pg_ot_array.append(pg_img_ot)

    original_name = os.path.basename(img)
    new_name = os.path.splitext(original_name)[0]+'_ot.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, pg_img_ot, header, overwrite=True)


#NGC6823    
ngc_o_array = []
ngc_ot_array = []

for img in ngc_files:
    ngc_img, header = fits.getdata(img, header=True)
    ccd = CCDData(ngc_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    ngc_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    ngc_img_ot = ngc_img_o[:, 53:2101]

    ngc_o_array.append(ngc_img_o)
    ngc_ot_array.append(ngc_img_ot)

    original_name = os.path.basename(img)
    new_name = os.path.splitext(original_name)[0]+'_ot.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, ngc_img_ot, header, overwrite=True)


# print(len(bias_o_array))
# print(len(bias_ot_array))
# print(len(flat_o_array))
# print(len(flat_ot_array))
# print(len(dark_o_array))
# print(len(dark_ot_array))
# print(len(pg_o_array))
# print(len(pg_ot_array))
# print(len(ngc_o_array))
# print(len(ngc_ot_array))
                                

## Bias Correction <a class="anchor" id="otz"></a>

In [ ]:
##CHARACTERIZE BIASES TO SEE IF YOU WANT TO COMBINE TO MASTER BIAS

interval = ZScaleInterval()

row_selected = 2

bias_time = np.arange(0, len(bias_ot_array), 1)
bias_mean = np.array([np.mean(bias[row_selected, :]) for bias in bias_ot_array])
bias_median = np.array([np.median(bias[row_selected, :]) for bias in bias_ot_array])
bias_std = np.array([np.std(bias[row_selected, :]) for bias in bias_ot_array])

pg_time = np.arange(0, len(pg_ot_array), 1)
pg_mean = np.array([np.mean(img[row_selected, :]) for img in pg_ot_array])
pg_median = np.array([np.median(img[row_selected, :]) for img in pg_ot_array])
pg_std = np.array([np.std(img[row_selected, :]) for img in pg_ot_array])


plt.plot(pg_time, pg_mean, label='Mean')
plt.plot(pg_time, pg_median, label='Median')
#plt.plot(bias_time, bias_std, label='Standard Deviation')
plt.xlabel('File Order (shows time progression through night)')
plt.ylabel('Average Pixel Value (for row '+str(row_selected+1)+')')
plt.title('Pixel Value throughout Night for PG1633 (after overscan subtraction and trimming)')
plt.legend()
plt.show()

plt.plot(bias_time, bias_mean, label='Mean')
plt.plot(bias_time, bias_median, label='Median')
#plt.plot(bias_time, bias_std, label='Standard Deviation')
plt.xticks(bias_time)
plt.xlabel('File Order (shows time progression through night)')
plt.ylabel('Average Pixel Value (for row '+str(row_selected+1)+')')
plt.title('Pixel Value throughout Night for Biases (after overscan subtraction and trimming)')
plt.legend()
plt.show()

In [ ]:
##CREATE MASTER BIAS

bias_ot_files = sorted(
    glob.glob(reduced_dir+'a09[3-9]_ot.fits') + #93-99
    glob.glob(reduced_dir+'a10[0-9]_ot.fits') + #100-109
    glob.glob(reduced_dir+'a11[0-1]_ot.fits')   #110-111
)
print('The number of biases we have is:', len(bias_ot_files))

master_bias = ccdp.combine(bias_ot_files,
                           unit='adu',
                           method='average',
                           sigma_clip=False)
master_bias_array = np.array(master_bias)
output_path = os.path.join(output_dir, 'master_bias.fits')
fits.writeto(output_path, master_bias_array, overwrite=True)

bias_column_range = np.arange(0, master_bias_array.shape[1], 1)
bias_means = []
for column in bias_column_range:
    mean = np.mean(master_bias_array[:, column])
    bias_means.append(mean)

pg_file = reduced_dir+'a081_ot.fits'
ngc_file = reduced_dir+'a156_ot.fits'
pg_image = fits.getdata(pg_file)
ngc_image = fits.getdata(ngc_file)
pg_column_range = np.arange(0, pg_image.shape[1], 1) #redundant because images same size, but doing for completeness
ngc_column_range = np.arange(0, ngc_image.shape[1], 1) #redundant because images same size, but doing for completeness
pg_means = []
ngc_means = []
for column in pg_column_range:
    mean = np.mean(pg_image[:, column])
    pg_means.append(mean)
for column in ngc_column_range:
    mean = np.mean(ngc_image[:, column])
    ngc_means.append(mean)




master_bias_file = reduced_dir+'master_bias.fits'
master_bias_image = fits.getdata(master_bias_file)
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(master_bias_image)
plt.imshow(master_bias_image, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title('Master Bias')
plt.show()

# for bias in bias_ot_array:
#     vmin, vmax = interval.get_limits(master_bias)
#     plt.imshow(master_bias, vmin=vmin, vmax=vmax)
#     plt.grid(False)
#     plt.xticks([])
#     plt.yticks([])
#     plt.show()

plt.scatter(bias_column_range, bias_means, s=1)
plt.xlabel('Column Number')
plt.ylabel('Average Pixel Value')
plt.title('Average Pixel Value by Column for Master Bias')
plt.show()

plt.scatter(pg_column_range, pg_means, s=1)
plt.xlabel('Column Number')
plt.ylabel('Average Pixel Value')
plt.title('Average Pixel Value by Column for PG Image: ' + str(pg_file[-12:]))
plt.ylim(0, 30)
plt.show()

plt.scatter(ngc_column_range, ngc_means, s=1)
plt.xlabel('Column Number')
plt.ylabel('Average Pixel Value')
plt.title('Average Pixel Value by Column for NGC Image: ' + str(ngc_file[-12:]))
plt.ylim(0, 250)
plt.show()



In [ ]:
## BIAS CORRECTION TO FILES (this is the same loop over again for different glob lists!)

#BIASES     
bias_o_array = []
bias_ot_array = []
bias_otz_array = []

for bias in bias_files:
    bias_img, header = fits.getdata(bias, header=True)
    ccd = CCDData(bias_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    bias_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    bias_img_ot = bias_img_o[:, 53:2101]
    bias_img_otz = bias_img_ot - master_bias_array

    bias_o_array.append(bias_img_o)
    bias_ot_array.append(bias_img_ot)
    bias_otz_array.append(bias_img_otz)

    original_name = os.path.basename(bias)
    new_name = os.path.splitext(original_name)[0]+'_otz.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, bias_img_otz, header, overwrite=True)




#FLATS     
flat_o_array = []
flat_ot_array = []
flat_otz_array = []

for flat in flat_files:
    flat_img, header = fits.getdata(flat, header=True)
    ccd = CCDData(flat_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    flat_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    flat_img_ot = flat_img_o[:, 53:2101]
    flat_img_otz = flat_img_ot - master_bias_array

    flat_o_array.append(flat_img_o)
    flat_ot_array.append(flat_img_ot)
    flat_otz_array.append(flat_img_otz)

    original_name = os.path.basename(flat)
    new_name = os.path.splitext(original_name)[0]+'_otz.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, flat_img_otz, header, overwrite=True)


#DARKS     
dark_o_array = []
dark_ot_array = []
dark_otz_array = []

for dark in dark_files:
    dark_img, header = fits.getdata(dark, header=True)
    ccd = CCDData(dark_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    dark_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    dark_img_ot = dark_img_o[:, 53:2101]
    dark_img_otz = dark_img_ot - master_bias_array

    dark_o_array.append(dark_img_o)
    dark_ot_array.append(dark_img_ot)
    dark_otz_array.append(dark_img_otz)

    original_name = os.path.basename(dark)
    new_name = os.path.splitext(original_name)[0]+'_otz.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, dark_img_otz, header, overwrite=True)


#PG1633    
pg_o_array = []
pg_ot_array = []
pg_otz_array = []

for img in pg_files:
    pg_img, header = fits.getdata(img, header=True)
    ccd = CCDData(pg_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    pg_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    pg_img_ot = pg_img_o[:, 53:2101]
    pg_img_otz = pg_img_ot - master_bias_array

    pg_o_array.append(pg_img_o)
    pg_ot_array.append(pg_img_ot)
    pg_otz_array.append(pg_img_otz)

    original_name = os.path.basename(img)
    new_name = os.path.splitext(original_name)[0]+'_otz.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, pg_img_otz, header, overwrite=True)


#NGC6823    
ngc_o_array = []
ngc_ot_array = []
ngc_otz_array = []

for img in ngc_files:
    ngc_img, header = fits.getdata(img, header=True)
    ccd = CCDData(ngc_img, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    overscan_ccd = CCDData(overscan, unit=ccd.unit)
    
    ngc_img_o = ccdp.subtract_overscan(ccd, 
                                overscan = overscan_ccd, 
                                overscan_axis = 1,
                                model = Polynomial1D(degree=3)
                                ).data
    
    ngc_img_ot = ngc_img_o[:, 53:2101]
    ngc_img_otz = ngc_img_ot - master_bias_array

    ngc_o_array.append(ngc_img_o)
    ngc_ot_array.append(ngc_img_ot)
    ngc_otz_array.append(ngc_img_otz)

    original_name = os.path.basename(img)
    new_name = os.path.splitext(original_name)[0]+'_otz.fits'
    output_path = os.path.join(output_dir, new_name)
    fits.writeto(output_path, ngc_img_otz, header, overwrite=True)


# print(len(bias_o_array))
# print(len(bias_ot_array))
# print(len(flat_o_array))
# print(len(flat_ot_array))
# print(len(dark_o_array))
# print(len(dark_ot_array))
# print(len(pg_o_array))
# print(len(pg_ot_array))
# print(len(ngc_o_array))
# print(len(ngc_ot_array))

## Darks Analysis <a class="anchor" id="darks"></a>

In [ ]:
## CALCULATE DARK CURRENT

avg_pixel_array = []
exp_time_array = []
dark_current_array = []
tot_pixel_array = []
gain = 2.5

for dark in dark_otz_files:
    dark_otz_img, hdr = fits.getdata(dark, header=True)

    tot_pixel_val = np.sum(dark_otz_img)
    avg_pixel_val = np.median(dark_otz_img)*gain
    exptime = hdr['EXPTIME']
    dark_current = avg_pixel_val/exptime

    print('Dark File:', dark[-13:-9],
          'Exposure Time:', exptime,
          'Median Pixel Value:', avg_pixel_val,
          'Dark Current:', dark_current,
          'Total Pixel Val:', tot_pixel_val)

    avg_pixel_array.append(avg_pixel_val)
    exp_time_array.append(exptime)
    dark_current_array.append(dark_current)
    tot_pixel_array.append(tot_pixel_val)

dark_file = np.arange(0, len(dark_otz_files), 1)
plt.plot(dark_file, dark_current_array)
plt.xlabel('Dark File')
plt.ylabel(r'Dark Current')
plt.title('Dark Current by Dark File')
plt.show()

plt.plot(dark_file, avg_pixel_array)
plt.xlabel('Dark File')
plt.ylabel(r'Median Pixel Value')
plt.title('Median Pixel Value by Dark File')
plt.show()

plt.scatter(exp_time_array, tot_pixel_array)
plt.xlabel('Exposure Time (s)')
plt.ylabel('Total Pixel Value')

In [ ]:
## CREATE MASTER DARK 

interval = ZScaleInterval()

avg_darks = ccdp.combine(dark_otz_files,
                           unit='adu',
                           method='average',
                           sigma_clip=False)
avg_darks_rms = np.sqrt(np.mean(np.array(avg_darks)**2))
avg_darks_std = np.std(avg_darks)
print('RMS for Straight Average of All Darks', avg_darks_rms)
print('Standard Deviation for Straight Average of All Darks', avg_darks_std)

vmin, vmax = interval.get_limits(avg_darks)
plt.imshow(avg_darks, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title('Straight Average of All Darks')
plt.show()


median_darks = ccdp.combine(dark_otz_files,
                           unit='adu',
                           method='median',
                           sigma_clip=False)
median_darks_rms = np.sqrt(np.mean(np.array(median_darks)**2))
median_darks_std = np.std(median_darks)
print('RMS for Straight Median of All Darks', median_darks_rms)
print('Standard Deviation for Straight Median of All Darks', median_darks_std)

vmin, vmax = interval.get_limits(median_darks)
plt.imshow(median_darks, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title('Straight Median of All Darks')
plt.show()


avg_darks_sigclip = ccdp.combine(dark_otz_files,
                                 unit='adu',
                                 method='average',
                                 sigma_clip=True,
                                 sigma_clip_low_thresh=3,
                                 sigma_clip_high_thresh=3,
                                 sigma_clip_func=np.ma.mean,
                                 sigma_clip_dev_func=mad_std
                                 )
avg_darks_sigclip_rms = np.sqrt(np.nanmean(np.array(avg_darks_sigclip)**2))
avg_darks_sigclip_std = np.nanstd(avg_darks_sigclip)
print('RMS for Average of All Darks (3sigma-clipped)', avg_darks_sigclip_rms)
print('Standard Deviation for Average of All Darks (3sigma-clipped)', avg_darks_sigclip_std)

vmin, vmax = interval.get_limits(avg_darks_sigclip)
plt.imshow(avg_darks_sigclip, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title(r'Average of All Darks ($3{\sigma}$-clipped)')
plt.show()


median_darks_sigclip = ccdp.combine(dark_otz_files,
                                 unit='adu',
                                 method='median',
                                 sigma_clip=True,
                                 sigma_clip_low_thresh=3,
                                 sigma_clip_high_thresh=3,
                                 sigma_clip_func=np.ma.mean,
                                 sigma_clip_dev_func=mad_std
                                 )
median_darks_sigclip_rms = np.sqrt(np.nanmean(np.array(median_darks_sigclip)**2))
median_darks_sigclip_std = np.nanstd(median_darks_sigclip)
print('RMS for Median of All Darks (3sigma-clipped)', median_darks_sigclip_rms)
print('Standard Deviation for Median of All Darks (3sigma-clipped)', median_darks_sigclip_std)

vmin, vmax = interval.get_limits(median_darks_sigclip)
plt.imshow(median_darks_sigclip, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title(r'Median of All Darks ($3{\sigma}$-clipped)')
plt.show()



output_path = os.path.join(output_dir, 'master_dark.fits')
fits.writeto(output_path, np.array(avg_darks_sigclip), overwrite=True)

## DS9 Notes <a class="anchor" id="ds9"></a>

In DS9, you can view the header by opening the fits file and click the header option! This is a really helpful way to sanity check your results on python (or at least it is for me!). Also, you should play around with what the file looks like in DS9 with different scales. It's usually very helpful to view using a log scale. 